### RAG PIPELINE- data ingestion to vector db pipeline


In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [7]:
### read all the pdf inside the dir

def process_all_pdf(pdf_dir):
    all_doc = []
    pdf_dir = Path(pdf_dir)

    #find all pdf
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} pdf files to process")

    for pdf_file in pdf_files:
        print(f"\nprocessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            docs = loader.load()

            #add sourver information of meta
            for doc in docs:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = "pdf"

            all_doc.extend(docs)
            print(f" loaded {len(docs)} pages")

        except Exception as e:
            print(f" x error: {e}")

    print(f"\n Total doc loaded: {len(all_doc)} pages")
    return all_doc

all_pdf_doc = process_all_pdf("../data")

found 3 pdf files to process

processing: CV.pdf
 loaded 2 pages

processing: jurnal.pdf
 loaded 6 pages

processing: Pitch.pdf
 loaded 71 pages

 Total doc loaded: 79 pages


In [8]:
all_pdf_doc

[Document(metadata={'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2026-03-31T15:32:00+00:00', 'title': 'Resume', 'author': 'Kinobi', 'subject': 'Resume', 'keywords': 'Resume', 'source': '..\\data\\pdf_files\\CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'CV.pdf', 'file_type': 'pdf'}, page_content='AZIZU AHMAD ROZAKI RIYANTO\n(+62)82136359686 | azizu.rozaki@gmail.com | https://www.linkedin.com/in/azizuahmad/ | https://azizuahmad.github.io/Portfolio-\nAzizuAhmad/\nSemarang City, Central Java, Indonesia\nData Analyst with 1 year of experience specializing in transforming complex datasets into actionable business insights using Power BI, \nTableau, and Excel. Deeply passionate about AI, with hands-on expertise in Python, SQL, and TensorFlow to build predictive models, \nNLP, and Image Processing solutions. Proven ability to bridge the gap between technical data science and strategic decision-making to \ndrive organizational growth.\nWork Experience

In [10]:
### text splitting get into chunks

def split_doc(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n",""," ","\n"]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)}")

    #show example of chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [11]:
chunks=split_doc(all_pdf_doc)
print(chunks)

Split 79 documents into 117

Example chunk:
Content: AZIZU AHMAD ROZAKI RIYANTO
(+62)82136359686 | azizu.rozaki@gmail.com | https://www.linkedin.com/in/azizuahmad/ | https://azizuahmad.github.io/Portfolio-
AzizuAhmad/
Semarang City, Central Java, Indone...
Metadata: {'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2026-03-31T15:32:00+00:00', 'title': 'Resume', 'author': 'Kinobi', 'subject': 'Resume', 'keywords': 'Resume', 'source': '..\\data\\pdf_files\\CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'CV.pdf', 'file_type': 'pdf'}
[Document(metadata={'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2026-03-31T15:32:00+00:00', 'title': 'Resume', 'author': 'Kinobi', 'subject': 'Resume', 'keywords': 'Resume', 'source': '..\\data\\pdf_files\\CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'CV.pdf', 'file_type': 'pdf'}, page_content='AZIZU AHMAD ROZAKI RIYANTO\n(+62)82136359686 | azizu.rozaki@gmail.com | https://www.l

### embedding and vectorstoredb

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

e:\AI Engineer\Ai Engineer 2026\RAG Langchain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
class EmbeddingManager:
    #handle docc embedding generation using sentencetransformer
    def __init__(self,model_name: str ="all-MiniLM-L6-v2"):
        """
            initial the embedding manager

            args:
            model_name = HuggingFace
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self): #_ for private func
        """load the sentenceTransfo model"""
        try:
            print(f"loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded succesfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        generate embedding for a little of text

        args:
            texts:List of text string to embed
        
        returns:
            numpy array of embedding with shape (len(texts),embedding_dim)
        """

        if not self.model:
            raise ValueError("Model Not Loaded")

        print(f"Generating embedding for {len(texts)} texts")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

    # def get_embedding_dimension(self) -> int:
    #     """get the embedding dimension of the model"""
    #     if not self.model:
    #         raise ValueError("Model not loaded")
        
    #     return self.model.get_sentence_embedding_dimension()

## intitial the embedding managerr
embedding_manager = EmbeddingManager()
embedding_manager


loading embedding model: all-MiniLM-L6-v2


e:\AI Engineer\Ai Engineer 2026\RAG Langchain\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Acer\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4467.19it/s]
BertModel 

Model loaded succesfully. Embedding dimension: 384


### Vector Store


In [ ]:
### vestor store

from typing import Any


class VectorStore:
    """Manage document embedding in a ChromeDB vector Store"""

    def __init__(self, collection_name: str = "pdf_doccuments", persist_directory: str = "../data/vector_store"):
        """
        init the vector store

        args:
            collection_name: name of the chromedb collection
            persist_directory: Directory to persist vector store
        """
        self.collection_name = collection_name
        self.persist_direcctory= persist_directory
        self.client = None
        self.coleection= None
        self._initialize_store()

    def _initialize_store(self): #private func
        """initial chromadb client and coll"""
        try:
            #create persistent chromadb client
            os.makedirs(self.persist_direcctory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_direcctory)

            #get or creaate collection
            self.collection = self.client.get_or_create_collection(
                name= self.collection_name,
                metadata= {"description":"PDF document embedding for RAG"}
            )
            print(f"vector store initialized. collection: {self.collection_name}")
            print(f"Existing document is collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error init vector store {e}")
            raise

    def add_documents(self, documents: List[any], embeddings: np.ndarray):
        """
        add doc and their embedding to the vector store

        args:
            documents: list of langchain doc
            embedding: Corresponding embed for doc
        
        """
        if len(documents) != len(embeddings):
            raise ValueError(f"Number of documents must match number of embeddings")

        print(f"adding {len(documents)} documents to vector store..")

        #prepare data for chromadb
        ids= []
        metadatas= []
        documents_text= []
        embeddings_list= []
        
        for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
            #generate unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata= dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #doc content
            documents_text.append(doc.page_content)

            #embedding
            embeddings_list.append(embedding.tolist())
        
        #add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"succesfully added {len(documents)} documents to vector store")
            print(f"total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"error adding doc to vector store {e}")
            raise

vectorstore= VectorStore()
vectorstore

    

vector store initialized. collection: pdf_doccuments
Existing document is collection: 0


In [32]:
chunks

[Document(metadata={'producer': 'pdfmake', 'creator': 'pdfmake', 'creationdate': '2026-03-31T15:32:00+00:00', 'title': 'Resume', 'author': 'Kinobi', 'subject': 'Resume', 'keywords': 'Resume', 'source': '..\\data\\pdf_files\\CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'CV.pdf', 'file_type': 'pdf'}, page_content='AZIZU AHMAD ROZAKI RIYANTO\n(+62)82136359686 | azizu.rozaki@gmail.com | https://www.linkedin.com/in/azizuahmad/ | https://azizuahmad.github.io/Portfolio-\nAzizuAhmad/\nSemarang City, Central Java, Indonesia\nData Analyst with 1 year of experience specializing in transforming complex datasets into actionable business insights using Power BI, \nTableau, and Excel. Deeply passionate about AI, with hands-on expertise in Python, SQL, and TensorFlow to build predictive models, \nNLP, and Image Processing solutions. Proven ability to bridge the gap between technical data science and strategic decision-making to \ndrive organizational growth.\nWork Experience

In [33]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]

### geneerate the embedding
embeddings= embedding_manager.generate_embeddings(texts)

### store into vector db
vectorstore.add_documents(chunks,embeddings)

Generating embedding for 117 texts


Batches: 100%|██████████| 4/4 [00:01<00:00,  2.07it/s]


Generated embedding with shape: (117, 384)
adding 117 documents to vector store..
succesfully added 117 documents to vector store
total documents in collection: 117


### Retriever pipeline from vectorstore


In [ ]:
class RAGRetriever:
    """handles query based retireval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Init the retriever

        args:
            vector_store: vector store containing document embedding
            embedding_manager: manager for generatig query embedding
        """
        self.vector_store = vector_store
        self.embedding_manager= embedding_manager

    def retrieve(self, query: str, top_k: int=5, score_threshold: float=0.0)-> List[Dict[str,Any]]:
        """
        Retrieve relevant document for a query

        args:
            query: the search query
            top_k: number of top result to return
            score_thershold: minimum similarity score threshold

        returns:
            listt of dictionaries containing retrieved document and metada
        """
        print(f"Retrieving document for query {query}")
        print(f"Top K: {top_k}, score threshold: {score_threshold}")

        #generate query emebdding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        #search in venctor store
        try:
            results = self.vector_store.collection.query(
                query_embeddings= [query_embedding.tolist()],
                n_results= top_k
            )

            #proses result
            retrieved_docs= []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #convert distanace to similarity score (chroma db uses cosidne distancee)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("no doc found")
            
            return retrieved_docs
        
        except Exception as e:
            print(f"error during returieval: {e}")
            return []


rag_retriever= RAGRetriever(vectorstore,embedding_manager)


    

In [35]:
rag_retriever

In [36]:
rag_retriever.retrieve("what is technical analysis")

Retrieving document for querywhat is technical analysis
Top K: 5, score threshold: 0.0
Generating embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.01it/s]

Generated embedding with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_db3a285e_49',
  'content': 'Technical Analysis\nTechnical analysis is the study of market action, primarily through the\nuse of charts, for the purpose of forecasting future price trends.\nThe Principles of TA\n1. Market discounts everything\n2. Price moves in trends \n3.History tends to repeat itself',
  'metadata': {'title': 'Pitch Deck',
   'doc_index': 49,
   'source': '..\\data\\pdf_files\\Pitch.pdf',
   'creator': 'Canva',
   'keywords': 'DAG9biW_CUI,BAG8CV3KdSo,0',
   'source_file': 'Pitch.pdf',
   'creationdate': '2026-01-08T05:48:17+00:00',
   'moddate': '2026-01-08T05:48:06+00:00',
   'page': 2,
   'page_label': '3',
   'producer': 'Canva',
   'file_type': 'pdf',
   'total_pages': 71,
   'content_length': 268,
   'author': 'leonardosusanto22'},
  'similarity_score': 0.6278101801872253,
  'distance': 0.37218981981277466,
  'rank': 1}]

In [43]:
rag_retriever.retrieve("Technical analysis is the study of market action, primarily through theuse of charts, for the purpose of forecasting future price trends")


Retrieving document for queryTechnical analysis is the study of market action, primarily through theuse of charts, for the purpose of forecasting future price trends
Top K: 5, score threshold: 0.0
Generating embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.38it/s]

Generated embedding with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_db3a285e_49',
  'content': 'Technical Analysis\nTechnical analysis is the study of market action, primarily through the\nuse of charts, for the purpose of forecasting future price trends.\nThe Principles of TA\n1. Market discounts everything\n2. Price moves in trends \n3.History tends to repeat itself',
  'metadata': {'file_type': 'pdf',
   'doc_index': 49,
   'source_file': 'Pitch.pdf',
   'content_length': 268,
   'creator': 'Canva',
   'producer': 'Canva',
   'moddate': '2026-01-08T05:48:06+00:00',
   'keywords': 'DAG9biW_CUI,BAG8CV3KdSo,0',
   'author': 'leonardosusanto22',
   'total_pages': 71,
   'title': 'Pitch Deck',
   'page_label': '3',
   'page': 2,
   'creationdate': '2026-01-08T05:48:17+00:00',
   'source': '..\\data\\pdf_files\\Pitch.pdf'},
  'similarity_score': 0.9202497601509094,
  'distance': 0.07975023984909058,
  'rank': 1},
 {'id': 'doc_562dca14_59',
  'content': 'Charts',
  'metadata': {'moddate': '2026-01-08T05:48:06+00:00',
   'title': 'Pitch Deck',


### integration vectordb contect pipeline with LLM output

In [48]:
### simple rag pipeline with openai

from langchain_openai import OpenAI 
import os
from dotenv import load_dotenv
load_dotenv()

# init the openai llm
openai_api_key = os.getenv("OPENAI_APIKEY")

llm = OpenAI(api_key=openai_api_key,model_name="gpt-4o-mini",temperature=0.1, max_tokens=1024)

## 2. simple rag fucntion: retrieve context + generate resppons
def rag_simple(query, retriever,llm,top_k=3):
    results= retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "no relevant contect found to answer the question"
    
    ###generate the  answer using openai llm
    prompt=f"""Use the following context to answer the question concisely.
        context:
        {context}

        Question: {query}

        Answer:
        """

    response= llm.invoke([prompt.format(context=context,query=query)])
    return response

In [50]:
answer=rag_simple("Explain about DREAMS (Dinus Research Group for AI in Medical Science)?",rag_retriever,llm)
print(answer)

Retrieving document for queryExplain about DREAMS (Dinus Research Group for AI in Medical Science)?
Top K: 3, score threshold: 0.0
Generating embedding for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.83it/s]

Generated embedding with shape: (1, 384)
Retrieved 1 documents (after filtering)


 DREAMS (Dinus Research Group for AI in Medical Science) is a research initiative at Universitas Dian Nuswantoro that involves collaboration among students, lecturers, and professors. The focus of the group is on the health sector, aiming to develop computationally efficient and highly accurate models using various Machine Learning and Deep Learning algorithms. The research is conducted from October 2023 to February 2025.
